# Датасет Avito Context Ad Clicks
### Важное замечание: анализ в этом ноутбуке проводился на полном датасете, но так как его размер слишком большой чтобы приложить его к файлам в архиве, вместо полных файлов сохранены выборки из них с меньшим размером
### Полный датасет можно изучить и скачать на платформе kaggle по ссылке: https://www.kaggle.com/competitions/avito-context-ad-clicks/data

In [5]:
import duckdb
import pandas as pd
import os

## 1. Загрузка данных

### Максимальный размер исходных файлов почти 13 гб, работать с ними в DataFrame долго и не эффективно, поэтому загружаем файлы датасета в базу данных. Для работы была выбрана DuckDB

### Датасет включает в себя 12 файлов, но в рамках задания полезную информацию содержат 4 таблицы:
#### - `ads_info` - объявления с заголовками и категориями
#### - `search_info` - поисковые сессии пользователей
#### - `train_search_stream` - показы объявлений в поиске (impressions)
#### - `visits_stream` - визиты на страницы объявлений

In [9]:
conn = duckdb.connect('avito_context.duckdb')

In [2]:
files = {
    'ads_info': 'data/AdsInfo.tsv',
    'search_info': 'data/SearchInfo.tsv',
    'train_search_stream': 'data/trainSearchStream.tsv',
    'visits_stream': 'data/VisitsStream.tsv'
}

for table, file in files.items():
    print(f'\nзагрузка {file} в {table}')

    conn.execute(f'''
        CREATE OR REPLACE TABLE {table} AS
        SELECT * FROM read_csv_auto(
            '{file}',
            delim='\t',
            header=True,
            AUTO_DETECT=True
        )
    ''')
    count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'загружено {count:,} строк')
print('бд успешно инициализирована')


загрузка data/AdsInfo.tsv в ads_info


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

загружено 36,893,298 строк

загрузка data/SearchInfo.tsv в search_info


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

загружено 112,159,462 строк

загрузка data/trainSearchStream.tsv в train_search_stream


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

загружено 392,356,948 строк

загрузка data/VisitsStream.tsv в visits_stream


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

загружено 286,821,375 строк
бд успешно инициализирована


In [3]:
conn = duckdb.connect('avito_context.duckdb')

## 2. Поиск категории автозапчастей

### Тест касается раздела **Автозапчасти** на Фарпосте, поэтому нужно найти аналогичную категорию в датасете Авито, чтобы извлечь реалистичные параметры поведения именно этой категории пользователей

### Вместо того чтобы смотреть на весь сайт, нужно отфильтровать объявления по ключевым словам, характерным для автозапчастей: `запчаст`, `бампер`, `капот`, `крыл`, `фар`, `амортизатор`, `шин`, `диск` и т.д.

### Запрос группирует результаты по `CategoryID` и считает количество объявлений в каждой категории. Это даст топ-категории по объёму объявлений с автозапчастями

In [11]:
query = '''
    SELECT CategoryID AS category_id,
        COUNT(*) AS total_ads,
        FIRST(Title) AS example_title
    FROM ads_info
    WHERE title ILIKE '%запчаст%' 
       OR title ILIKE '%бампер%'
       OR title ILIKE '%капот%'
       OR title ILIKE '%крыл%'
       OR title ILIKE '%фар%'
       OR title ILIKE '%фонар%'
       OR title ILIKE '%амортизатор%'
       OR title ILIKE '%стойк%'
       OR title ILIKE '%колодк%'
       OR title ILIKE '%двигател%'
       OR title ILIKE '%двс%'
       OR title ILIKE '%акпп%'
       OR title ILIKE '%шин%'
       OR title ILIKE '%диск%'
    GROUP BY category_id
    ORDER BY total_ads DESC
    LIMIT 10;
'''
df_categories = conn.execute(query).fetchdf()
df_categories

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,category_id,total_ads,example_title
0,34,1846485,Кузовные д. бмв E87 решетка бампера черн
1,27,87553,Стиральная машина Индезит (Indesit )
2,53,55494,Двигатели 230/240v 32w 1300/1550Rpm
3,47,29948,Машина Электро Джип
4,50,28637,Монтажная стойка талреп
5,250005,27719,Реализую шины диски
6,37,25876,BMW M3 E30 Street 1987 1/18 с дисками M-Style 21
7,29,24265,"Жёсткий диск ноутбука HDD SATA 500Гб 2.5"". DDR3L"
8,41,23863,Nokia N8 на запчасти или для ремонта
9,38,22704,Футляры швейных машин


## 3. Ручная проверка категорий

### Поиска по ключевым словам недостаточно. Например, слово `диск` может встречаться в объявлениях о музыке или компьютерных комплектующих, `стойк` - в строительстве. Поэтому для каждой из топ-категорий вручную посмотрим 10 первых заголовков, чтобы убедиться что категория действительно про автозапчасти.

In [12]:
query = '''
SELECT CategoryID as category_id,
Title as title
FROM ads_info
WHERE CategoryID == 34
LIMIT 10;
'''
df_category_34 = conn.execute(query).fetchdf()
df_category_34

,category_id,title
0,34,Передние брызговики Форд Фокус 2 родные
1,34,Поворотник R - Carina (20317)
2,34,ВАЗ дверь 2106 передняя левая
3,34,Форд фокус 2 зеркало Ford Focus 2
4,34,Фонари ауди а6 с5
5,34,Чип тюнинг от ледокола форд фокус 2 1.8
6,34,205 65 15c ContinentalConti Vanko Winter 2
7,34,Заднее стекло фольксваген Пасат B5/B5+
8,34,Бампер передний грунт под покраску lancer 9 06...
9,34,Продам коленвал двигатель 4G64


### Категория 34 - это Автозапчасти. Видим поворотники, зеркала, брызговики, бамперы, двигатели. Это всё прямые аналоги раздела Автозапчасти на Фарпосте. Это самая большая категория (1.8 млн объявлений) - берём её как основу.

### Для полноты проверяем остальные категории из топа, чтобы убедиться что они не подходят.

In [14]:
query = '''
SELECT CategoryID as category_id,
Title as title
FROM ads_info
WHERE CategoryID == 27
LIMIT 10;
'''
df_category_27 = conn.execute(query).fetchdf()
df_category_27

,category_id,title
0,27,Ardo Молокоотсос Calypso электрический
1,27,Отпариватель MS10748
2,27,Самовар электрический
3,27,Электрическая печь
4,27,Миксер Scarlett sc-045
5,27,Увлажнитель воздуха Bork
6,27,Хол-к Атлант Даем гарантию Доставка
7,27,Холодильник полюс
8,27,Стиральная машинка Elenberg
9,27,Холодильник


### Категория 27 - Бытовая техника (стиральные машины, холодильники, отпариватели). Слово `диск` сюда попало скорее всего через 'диски стиральных машин' или подобное.

In [13]:
query = '''
SELECT CategoryID as category_id,
Title as title
FROM ads_info
WHERE CategoryID == 53
LIMIT 10;
'''
df_category_53 = conn.execute(query).fetchdf()
df_category_53

,category_id,title
0,53,Дровокол
1,53,Лента полипропиленовая от производителя
2,53,"Горка пристенная ""Эдельвейс"" б/у"
3,53,Стоматологическое оборудование
4,53,Пнемогайковерт
5,53,"Питатель пластинчатый 10 м3, производительност..."
6,53,Продаю циркулярка малая бдс-4
7,53,Кабинет руководителя премиум-класса Han Италия
8,53,Спотр для автосервиса
9,53,"Контейнер, бытовка, вагончик"


### Категория 53 - Промышленное оборудование (электродвигатели, станки). Ключевое слово `двигател` дало ложное срабатывание - эти двигатели не автомобильные

In [15]:
query = '''
SELECT CategoryID as category_id,
Title as title
FROM ads_info
WHERE CategoryID == 47
LIMIT 10;
'''
df_category_47 = conn.execute(query).fetchdf()
df_category_47

,category_id,title
0,47,Светодиодный конструктор lite brix Подиум
1,47,Продам новую коляску прогулочную Inglesina
2,47,Качели graco silhouette
3,47,Автомобильное кресло
4,47,Детская мини-кухня
5,47,Продается телескоп 3-х кратное увеличение новый
6,47,Tiny love развивающий коврик
7,47,"Объемный пазл ""сказочный рождественский замок"""
8,47,Пушка
9,47,Детская кроватка


### Категория 47 - Детские товары** (коляски, качели, кресла). Слово `автомобильное кресло` дало попадание, но это детское автокресло, не запчасть

In [16]:
query = '''
SELECT CategoryID as category_id,
Title as title
FROM ads_info
WHERE CategoryID == 50
LIMIT 10;
'''
df_category_50 = conn.execute(query).fetchdf()
df_category_50

,category_id,title
0,50,Продам газовые баллоны
1,50,Винил сайдинг файнбер Стандарт Сандал Подольск
2,50,"Ламинат Berry Alloc, производство Бельгия"
3,50,Тумба-умывальник изогнутая
4,50,"Деревянные садовые беседки ""Романтик"""
5,50,Качественные окна- лучший выбор
6,50,Перегородки для офиса
7,50,Обои
8,50,Станок рейсмусовый Makita 2012 NB
9,50,Лопата штыковая тип 1 рельсовая сталь


### Категория 50 - Стройматериалы и инструменты (ламинат, окна, лопаты). Не подходит

In [17]:
query = '''
SELECT CategoryID as category_id,
Title as title
FROM ads_info
WHERE CategoryID == 250005
LIMIT 10;
'''
df_category_250005 = conn.execute(query).fetchdf()
df_category_250005

,category_id,title
0,250005,Душевые кабины
1,250005,Наружная реклама и полиграфия любых вида и диз...
2,250005,Сборка и установка мебели
3,250005,Бесплатные стрижки
4,250005,Автосервис на 24-школе
5,250005,Шугаринг готовимся к Весне
6,250005,Maникюр с покрытием шеллак(гель-лак)
7,250005,Натяжные потолки за 1 день
8,250005,Вагончики дачные и строительные в Брянске
9,250005,Автоэлектрик диагност. Грузовые и легковые. Выезд


### Категория 250005 - Услуги** (маникюр, натяжные потолки, автосервис). Хотя `автосервис` близко к теме, это услуги, а не товарные объявления. Не подходит

In [18]:
query = '''
SELECT CategoryID as category_id,
Title as title
FROM ads_info
WHERE CategoryID == 37
LIMIT 10;
'''
df_category_37 = conn.execute(query).fetchdf()
df_category_37

,category_id,title
0,37,"Приёмник трехпрограммный ""русь-201"""
1,37,Точная копия спец техники китая
2,37,Старые чернильницы
3,37,"Фарфоровая статуэтка ""Колхозница с лошадью""Гжель"
4,37,Значки гдр
5,37,Подсвечник
6,37,3-й Рейх. Почтовый конгресс. 1942 год. мн
7,37,Печатная машинка Любава в кейсе
8,37,"Икона Ветхозаветная Троица, овальная, 19 век"
9,37,Moody Blues виниловые пластинки 5000 шт винил


### Категория 37 - Коллекционирование (значки, иконы, пластинки, антиквариат). Не подходит

In [19]:
query = '''
SELECT CategoryID as category_id,
Title as title
FROM ads_info
WHERE CategoryID == 29
LIMIT 10;
'''
df_category_29 = conn.execute(query).fetchdf()
df_category_29

,category_id,title
0,29,ZTE zxdsl 831
1,29,GT210 1GB
2,29,Продаётся Видеокарта GeForce GTX 650 zotac
3,29,Блок питания ATX 500W Cooler Master eXtreme
4,29,Для тошиба а100
5,29,Видеокарта Palit GeForce GTX 650 Ti
6,29,Память DDR2 для ноутбука 1Гб
7,29,Коммутатор Allied Telesyn Rapier AT-RP24i
8,29,Радиозапчасти от судовой рлс
9,29,Блок питания Asus Benq Compaq HP LG Roverbook


### Категория 29 - Компьютерные комплектующие (видеокарты, блоки питания, память). Слово `диск` дало попадание через HDD

In [20]:
query = '''
SELECT CategoryID as category_id,
Title as title
FROM ads_info
WHERE CategoryID == 41
LIMIT 10;
'''
df_category_41 = conn.execute(query).fetchdf()
df_category_41

,category_id,title
0,41,Самсунг 525
1,41,Красивый федеральный поменяю на красивый город...
2,41,Lenovo S860
3,41,Корпус Apple iPhone 5C белый
4,41,Motorola D402 Silver Телефон беспроводной dect
5,41,Новый iPhone 5s золото продаю
6,41,Nokia e7
7,41,HTC Desire HD
8,41,"Автомобильное зарядное устройство iPod, iPhone"
9,41,iPhone


### Категория 41 - Телефоны (Samsung, iPhone, Nokia). Не подходит

In [21]:
query = '''
SELECT CategoryID as category_id,
Title as title
FROM ads_info
WHERE CategoryID == 38
LIMIT 10;
'''
df_category_38 = conn.execute(query).fetchdf()
df_category_38

,category_id,title
0,38,"""Столичные"", Большой театр. СССР. Редкие"
1,38,Кукла Попик
2,38,Угловой шкаф
3,38,Шкаф купе
4,38,Трюмо с зеркалом
5,38,Для долгого пользования
6,38,Продаю шкаф-купе
7,38,Шкаф Навесной Лира 1255
8,38,Два пуфа в коричневом цвете
9,38,"Украшение интерьера, сделаю на заказ"


### Категория 38 - Хобби и досуг** (пластинки, швейные машины, коллекции). Не подходит.

## **Вывод:** из всего топа только категория 34 соответствует Автозапчастям. Все дальнейшие расчёты будут проводится по ней

## 4. Ежедневный трафик: количество уникальных пользователей

### Нужно понять сколько пользователей ежедневно заходят в раздел Автозапчасти, чтобы задать реалистичный размер выборки для генерации данных.

### Считаем по `search_info`: уникальные `UserID` за каждый день в категории 34

In [24]:
query = '''
SELECT 
    SearchDate::DATE AS search_date,
    CategoryID AS category_id,
    COUNT(DISTINCT UserID) AS unique_users_per_day,
    COUNT(*) AS total_searches_per_day
FROM search_info
WHERE category_id = 34
GROUP BY 
    search_date, 
    category_id
ORDER BY 
    search_date;
'''
df_category_search_per_day = conn.execute(query).fetchdf()
df_category_search_per_day.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,search_date,category_id,unique_users_per_day,total_searches_per_day
0,2015-04-25,34,45645,344498
1,2015-04-26,34,50828,397176
2,2015-04-27,34,60740,445379
3,2015-04-28,34,59325,448834
4,2015-04-29,34,59089,459922


### Посмотрим на агрегированные значения. Нас интересует **медиана**, а не среднее, потому что  среднее чувствительно к выбросам (праздничные дни, технические сбои), медиана даёт более стабильную оценку 'обячного дня'

In [31]:
print(f'Количество дней: {df_category_search_per_day.shape[0]}')

Количество дней: 26


In [44]:
print(f'Среднее количество уникальных пользователей в день: {int(df_category_search_per_day['unique_users_per_day'].mean())}')
print(f'Количество уникальных пользователей в день (медиана): {int(df_category_search_per_day['unique_users_per_day'].median())}')
print(f'\nСреднее количество поисков в данной категории в день: {int(df_category_search_per_day['total_searches_per_day'].mean())}')
print(f'Количество поисков в данной категории в день (медиана): {int(df_category_search_per_day['total_searches_per_day'].median())}')

Среднее количество уникальных пользователей в день: 65015
Количество уникальных пользователей в день (медиана): 61544

Среднее количество поисков в данной категории в день: 467785
Количество поисков в данной категории в день (медиана): 447106


## 5. Количество просмотренных объявлений за сессию

### Следующий параметр для генерации - сколько объявлений в среднем просматривает пользователь за одну сессию. Это напрямую влияет на `offers_viewed_per_session` в нашем синтетическом датасете.

### Важно: так как здесь не зафиксированы сессии пользователя, упрощенно будем считать сессией 1 день 

### Используем `visits_stream` - таблицу визитов на страницы объявлений. Группируем по пользователю и дню (`view_date`), считаем количество визитов и количество уникальных объявлений

### **Важный нюанс:** `visits_stream` фиксирует только переходы на страницу объявления, то есть уже состоявшийся клик. Объявления, которые пользователь видел в списке но не кликнул, здесь не отражены. Иными словами, это не `offers_viewed`, а `offers_clicked` в нашей терминологии. 

In [32]:
query = '''
SELECT 
    v.ViewDate::DATE as view_date,
    a.CategoryID AS category_id,
    UserID as user_id,
    COUNT(*) AS total_views,
    COUNT(DISTINCT v.AdID) AS unique_ads_viewed
FROM visits_stream v
JOIN ads_info a ON v.AdID = a.AdID
WHERE a.CategoryID = 34
GROUP BY 
    view_date,
    user_id,
    category_id,
ORDER BY view_date;
'''
df_ads_viewed = conn.execute(query).fetchdf()
df_ads_viewed.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,view_date,category_id,user_id,total_views,unique_ads_viewed
0,2015-04-25,34,1756440,69,64
1,2015-04-25,34,2321746,5,5
2,2015-04-25,34,1888654,3,3
3,2015-04-25,34,464883,12,11
4,2015-04-25,34,1251168,2,2


In [46]:
print(f'Среднее количество всего просмотренных объявлений: {int(df_ads_viewed['total_views'].mean())}')
print(f'Количество всего просмотренных объявлений (медиана): {int(df_ads_viewed['total_views'].median())}')
print(f'\nСреднее количество уникальных просмотренных объявлений: {int(df_ads_viewed['unique_ads_viewed'].mean())}')
print(f'Количество уникальных просмотренных объявлений (медиана): {int(df_ads_viewed['unique_ads_viewed'].median())}')

Среднее количество всего просмотренных объявлений: 7
Количество всего просмотренных объявлений (медиана): 3

Среднее количество уникальных просмотренных объявлений: 6
Количество уникальных просмотренных объявлений (медиана): 3


## 6. Время, проведённое на странице категории

### Следующий параметр - `time_on_page`. Здесь мы сталкиваемся с принципиальным ограничением данных: прямого события 'закрыл страницу' не существует.

### Обычно считают время сессии как разницу между временем последнего события и временем первого. Мы делаем так:

#### 1. Берём время первого поискового запроса пользователя за день в категории 34 как начало сессии (`MIN(SearchDate)` из `search_info`)
#### 2. Берём время последнего просмотра объявления за тот же день как конец сессии (`MAX(ViewDate)` из `visits_stream`)
#### 3. Разница - это оценка длительности сессии

### **Нюансы:**
#### - Если пользователь искал, но не кликнул ни на одно объявление — он выпадет из выборки (нет `last_view_time`). Такие сессии не учитываются
#### - Если пользователь открыл объявление и ушел заниматься своими делами — это время посчитается как 'активное'. Чтобы отсечь такие аномалии, добавляем фильтр `duration < 10800` (3 часа)
#### - Фильтруем сессии с `duration > 0` - это убирает случаи когда поиск и просмотр объявления произошли в одну секунду (технические ошибки)

### Таким образом, оценка времени - это нижняя граница реального времени на странице для пользователей, которые хотя бы один раз кликнули на объявление

In [39]:
query = '''
WITH user_day_bounds AS (
    SELECT 
        UserID,
        SearchDate::DATE AS session_date,
        MIN(SearchDate) AS first_search_time,
        CAST(NULL AS TIMESTAMP) AS last_view_time
    FROM search_info
    WHERE CategoryID = 34
    GROUP BY UserID, session_date

    UNION ALL

    SELECT 
        v.UserID,
        v.ViewDate::DATE AS session_date,
        CAST(NULL AS TIMESTAMP) AS first_search_time,
        MAX(v.ViewDate) AS last_view_time
    FROM visits_stream v
    JOIN ads_info a ON v.AdID = a.AdID
    WHERE a.CategoryID = 34
    GROUP BY v.UserID, session_date
),

session_durations AS (
    SELECT 
        UserID,
        session_date,
        MIN(first_search_time) AS session_start,
        MAX(last_view_time) AS session_end,
        date_diff('second', MIN(first_search_time), MAX(last_view_time)) AS duration_seconds
    FROM user_day_bounds
    GROUP BY UserID, session_date
    HAVING MIN(first_search_time) IS NOT NULL 
       AND MAX(last_view_time) IS NOT NULL
       AND date_diff('second', MIN(first_search_time), MAX(last_view_time)) > 0
       AND date_diff('second', MIN(first_search_time), MAX(last_view_time)) < 10800
)

SELECT * FROM session_durations;
'''
df_time_per_day = conn.execute(query).fetchdf()
df_time_per_day.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,UserID,session_date,session_start,session_end,duration_seconds
0,951024,2015-04-29,2015-04-29 06:53:55,2015-04-29 07:24:26,1831
1,1744390,2015-05-05,2015-05-05 08:51:52,2015-05-05 09:05:34,822
2,1078228,2015-05-11,2015-05-11 12:17:02,2015-05-11 12:37:43,1241
3,1193448,2015-04-30,2015-04-30 08:46:33,2015-04-30 09:35:05,2912
4,898138,2015-05-04,2015-05-04 14:13:04,2015-05-04 15:17:24,3860


### Среднее и медиана сильно расходятся - это нормально для времени сессий. Небольшая часть пользователей проводит очень много времени и тянет среднее вверх. Медиана более показательна для 'типичного пользователя'. Именно медианное значение берём как основу для параметра `loc` при генерации `time_on_page`

In [47]:
print(f'Среднее количество времени проведенное на странице: {int(df_time_per_day['duration_seconds'].mean() / 60)} мин')
print(f'Количество времени проведенное на странице (медиана): {int(df_time_per_day['duration_seconds'].median() / 60)} мин')

Среднее количество времени проведенное на странице: 21 мин
Количество времени проведенное на странице (медиана): 7 мин


## 7. Количество объявлений, показанных за один поисковый запрос

### Последний параметр - сколько объявлений в среднем показывается пользователю за один поисковый запрос. Это поможет подобрать значение для  `offers_viewed_per_session`.

### Используем `train_search_stream` - это таблица показов. Каждая строка соответствует одному объявлению, показанному в одном поисковом результате

### **Важная оговорка:** в датасете залогированы только позиции 1, 2, 6, 7, 8 (согласно описанию соревнования). Это значит что реальное количество показов на страницу больше - мы видим только часть. Рсчёт ниже даёт **нижнюю оценку** для `offers_viewed`

In [49]:
query = '''
SELECT 
    s.SearchDate::DATE AS date,
    COUNT(*) AS total_impressions,
    COUNT(DISTINCT t.AdID) AS unique_ads_shown,
    COUNT(DISTINCT t.SearchID) AS total_searches
FROM train_search_stream t
JOIN ads_info a ON t.AdID = a.AdID
JOIN search_info s ON t.SearchID = s.SearchID
WHERE a.CategoryID = 34
GROUP BY date
ORDER BY date;
'''
df_ads_showed = conn.execute(query).fetchdf()
df_ads_showed.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,date,total_impressions,unique_ads_shown,total_searches
0,2015-04-25,1982400,300460,644502
1,2015-04-26,2283349,330417,745937
2,2015-04-27,2539013,382029,826212
3,2015-04-28,2535437,394434,816883
4,2015-04-29,2545270,402832,814819


### Считаем среднее число показов на один поисковый запрос. Это соотношение агрегатов по дням (а не среднее по строкам) - чтобы избежать смещения в дни с разным трафиком.

In [51]:
avg_ads_shown_per_search = int(df_ads_showed['total_impressions'].mean() / df_ads_showed['total_searches'].mean())
avg_unique_ads_shown_per_search = int(df_ads_showed['unique_ads_shown'].mean() / df_ads_showed['total_searches'].mean())

print(f'Среднее количество объявлений, просмотренное за 1 результат поиска: {avg_ads_shown_per_search}')
print(f'Среднее количество уникальных объявлений, просмотренное за 1 результат поиска: {avg_unique_ads_shown_per_search}')

Среднее количество объявлений, просмотренное за 1 результат поиска: 3
Среднее количество уникальных объявлений, просмотренное за 1 результат поиска: 0


## 8. Итоговые параметры для генерации

### На основе проведённого анализа получаем следующие реалистичные параметры:

| Параметр | Значение | Источник |
|---|---|---|
| Уникальных пользователей в день | ~медиана из cell 15 | search_info, кат. 34 |
| Сессий за 21 день | ~медиана × 21 | расчётное |
| `time_on_page` (медиана) | ~7 мин = 420 сек | visits_stream + search_info |
| `offers_viewed` за сессию | ~1-2 (клики) | visits_stream (нижняя оценка) |
| Объявлений на поисковый запрос | ~3 | train_search_stream (частичный лог) |

### **Ограничения датасета Авито как ориентира для Фарпоста:**
#### - Авито - федеральная платформа, Фарпост - региональная (Дальний Восток). Трафик на Фарпосте будет значительно ниже
#### - Датасет Авито датирован 2015 годом. Поведение пользователей с тех пор могло измениться

### Тем не менее **качественные паттерны** (соотношение кликов к просмотрам, распределение времени на странице, ежедневная вариативность трафика) используются как реалистичная основа для синтетических данных

## Дополнительно: CTR для контекстных объявлений в категории запчасти
### Среднее - 1.6%
### Медиана - 0.7%

In [7]:
query = '''
SELECT 
    ROUND(AVG(t.HistCTR), 5) AS avg_historical_ctr,
    ROUND(MEDIAN(t.HistCTR), 5) AS median_historical_ctr,
    COUNT(*) AS total_contextual_impressions
FROM train_search_stream t
JOIN ads_info a ON t.AdID = a.AdID
WHERE t.ObjectType = 3
  AND a.CategoryID = 34
  AND t.HistCTR IS NOT NULL;
'''
df_context_ad_ctr = conn.execute(query).fetchdf()
df_context_ad_ctr

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,avg_historical_ctr,median_historical_ctr,total_contextual_impressions
0,0.01587,0.00649,35404641


In [7]:
conn.close()

## 9. Сохранение выборок в файлы

In [11]:
os.makedirs('data/samples', exist_ok=True)

tables = {
    'ads_info': 10000,
    'search_info': 10000,
    'train_search_stream': 10000,
    'visits_stream': 10000,
}

for table, n in tables.items():
    conn.execute(f"""
        COPY (
            SELECT * FROM {table} USING SAMPLE {n} ROWS (reservoir, 42)
        ) TO 'data/samples/sample_{table}.csv' (HEADER, DELIMITER ',')
    """)
    print(f'sample_{table}.csv — {n} строк')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sample_ads_info.csv — 10000 строк


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sample_search_info.csv — 10000 строк


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sample_train_search_stream.csv — 10000 строк


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sample_visits_stream.csv — 10000 строк
